# EM proofreading — Phase A (annotate)

Skeleton-driven review of a whole cell over MICrONS minnie65, dropping tagged
proofreading annotations. **Read-only** here — annotate now, edit manually later
(Phase B), then re-enter on the new root id (Phase C).

**Review model:** glide a precomputed **local** EM+target layer (smooth, renders during
motion) → **pause** to drop into the live EM+segmentation and annotate → resume.

Run in the `em` env (`uv run --extra em jupyter lab`, or the `.venv` kernel); needs a CAVE
token at `~/.cloudvolume/secrets/cave-secret.json`.

## 1. Imports

In [12]:
%load_ext autoreload
%autoreload 2

import logging
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)  # silence CloudVolume S3/GCS pool noise

import proofreading.em as em
from proofreading.em.wal import WAL

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Connect to CAVE

`minnie65_public` is the read-only **sandbox** (no edits, no root changes);
`minnie65_phase3_v1` is the live, proofreadable datastack.

In [13]:
client = em.EMClient('minnie65_public')
#   live: em.EMClient('minnie65_phase3_v1', version=<materialization_version>)
print('datastack', client.datastack, '| materialization', client.mat_version)

datastack minnie65_public | materialization 1718


## 3. Start a session

Loads the L2 skeleton, captures the durable **seed supervoxel**, builds the viewer, and
opens/append-resumes the write-ahead log.

In [21]:
root_id = 864691135572530981   # example cell on minnie65_public
sess = em.ProofreadSession(
    client, root_id, wal_dir='./proofread_sessions',
    step_nm=1000.0,                 # camera node spacing along each branch
    orient_to_path=False,           # axis-aligned sections (fast); True = cross-section ⊥ neurite
    seconds_per_step=0.4,           # glide speed (seconds per node)
    preview_target_nm=8.0,        # preview mip target; auto-coarsens further for long branches
    preview_pad_nm=1500.0,          # context (nm) around the neurite in the preview
    preview_max_voxels=25_000_000,  # cap cutout size -> coarser mip on big branches (faster)
    cross_section_render_scale=1.0, # live layers (shown on pause) at full res
)
print('seed', sess.seed, '| branch paths', len(sess.tree.branch_paths), '|', sess.summary())

seed 111692652428576376 | branch paths 189 | {'to_review': 184, 'covered': 3, 'omitted': 2}


In [14]:
v, fly = em.tube_prototype(client, 864691135572530981, path_id=5)   # try a few path_ids
v 

tube[path_5_mip1_em]: 508/508 chunks  133 MB  10.0s
tube[path_5_mip1_tgt]: 501/501 chunks  131 MB  20.2s
tube served at http://localhost:50921: EM 133 MB + target 131 MB


http://localhost:50841/v/dfe116a5f206dedf40990683adbdbbc79b637741/

In [15]:
fly.play()

In [ ]:
fly.pause()

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 53419)
Traceback (most recent call last):
  File "/Users/wanqing.yu/.local/share/uv/python/cpython-3.12-macos-x86_64-none/lib/python3.12/http/server.py", line 731, in send_head
    f = open(path, 'rb')
        ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/Users/wanqing.yu/Projects/proofreading-demo/proofread_sessions/tube_cache/minnie65_public/864691135572530981/path_5_mip1/16_16_40/84352-84416_23424-23488_18272-18336'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/wanqing.yu/.local/share/uv/python/cpython-3.12-macos-x86_64-none/lib/python3.12/socketserver.py", line 697, in process_request_thread
    self.finish_request(request, client_address)
  File "/Users/wanqing.yu/.local/share/uv/python/cpython-3.12-macos-x86_64-none/lib/python3.12/socketserver.py", line 362, in finish_requ

## 4. Open the viewer

Open the URL in a browser — 4-panel layout (3 cross-sections + 3D). The target cell is
highlighted; reveal neighbors with `n`.

In [16]:
sess.viewer

http://localhost:62951/v/048064e95cfebe838f0380d8c138e983fdda9f83/

## 5. Review — glide, pause, annotate

The core loop. `review_next()` builds the branch's **local preview** (EM + red target) and
starts a **paused** glide.

- **Play** → continuous smooth glide over the precomputed layer (target visible while moving).
- **Pause** → live full-res EM + real segmentation paint at that spot; annotate.
- **Resume** → continue; advancing to a new branch rebuilds its preview.

Keys in the neuroglancer window (while paused):

| key | action |
|-----|--------|
| `m` | merge error |
| `s` | split error |
| `e` | extend |
| `q` | question |
| `n` | toggle the segment under the cursor (reveal/hide a neighbor) |
| `x` | mark the current branch reviewed (and advance) |

A `merge error` ends the branch early and **prunes its distal subtree** off the checklist.

In [17]:
fly = sess.review_next()    # builds the branch preview + starts a PAUSED glide
# fly.play() / fly.pause() / fly.reverse() / fly.step(1)
# ad-hoc live glide with no preview:  sess.review_path(pid, preview=False)
fly

preview: mip5 [256, 256, 160]nm  shape (151, 189, 191)  11 MB   EM 2.8s   mask[skeleton] 0.0s


In [13]:
fly.play()

In [14]:
fly.pause()

## 6. Control panel (optional)

Branch-path checklist + **Review** / **Mark done** / **Resolve supervoxels**. The
FlyThrough play/pause/step controls appear after **Review**.

In [6]:
sess.panel()

## 7. Checkpoint — resolve supervoxels

Annotations store the click `xyz` immediately (durable); supervoxels are derived in batch
from CloudVolume. Run at checkpoints / before stepping away.

In [ ]:
print('resolved', sess.resolve_supervoxels(), 'supervoxels')

## 8. Inspect the log

The append-only write-ahead log is the source of truth — replay it any time.

In [ ]:
state = WAL.load(sess.wal.path)
print('log:', sess.wal.path)
for a in state.annotations.values():
    print(f'  {a.tag:12s} xyz={[round(c) for c in a.xyz]} supervoxel={a.supervoxel}')
print('coverage:', sess.summary())

## 9. After edits — re-entry (Phase C)

Once you've performed the manual splits/merges the root id changes. Recover the current
root from the durable seed and start fresh: the same WAL resumes, prior coverage re-attaches
by L2 id, and edited regions fall back to `to_review`.

```python
new_root = client.current_root(sess.seed)
sess2 = em.ProofreadSession(client, new_root, wal_dir='./proofread_sessions')
sess2.summary()
```

## 10. Shutdown

In [ ]:
sess.close()

## Appendix — validate local-layer-in-motion (optional, one-time)

The review model assumes a **local** layer renders sharp *during* camera motion (unlike the
live graphene seg). Confirm it once in a **fresh kernel** (this starts its own neuroglancer
server, separate from the session above): run the cell, open the viewer, `spike_fly.play()`,
and watch the synthetic checkerboard while it moves. Then restart the kernel for the real
workflow.

- **Crisp during motion** → the approach holds.
- **Blurs / blanks while moving, sharp only when stopped** → tell Claude; we pivot to the microviewer fallback.

In [ ]:
import proofreading.em as em
spike_viewer, spike_fly = em.localvolume_spike()   # synthetic; no CAVE needed
spike_viewer

In [ ]:
spike_fly.play()    # watch the checker while it moves; spike_fly.pause() to stop